# EDA — RetailRocket ecommerce dataset

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 4)
sns.set_style('whitegrid')

DATA_DIR = '../data'

In [ ]:
events = pd.read_csv(os.path.join(DATA_DIR, 'events.csv'))
events['timestamp'] = pd.to_datetime(events['timestamp'], unit='ms')
print('строк:', len(events))
print()
print('пропуски:')
print(events.isna().sum())
events.head()

## Типы событий и воронка

In [ ]:
cnt = events['event'].value_counts()
print(cnt)
print()
print('% от view:')
print((cnt / cnt['view'] * 100).round(2))

In [ ]:
fig, ax = plt.subplots()
ax.bar(cnt.index, cnt.values, color=['steelblue', 'orange', 'green'])
for i, (k, v) in enumerate(cnt.items()):
    ax.text(i, v + 5000, f'{v:,}', ha='center', fontsize=10)
ax.set_title('количество событий по типу')
ax.set_ylabel('событий')
plt.tight_layout()
plt.show()

In [ ]:
# конверсии между ступенями воронки
uniq_v = events[events['event'] == 'view']['visitorid'].nunique()
uniq_a = events[events['event'] == 'addtocart']['visitorid'].nunique()
uniq_t = events[events['event'] == 'transaction']['visitorid'].nunique()

print(f'пользователей с хотя бы одним view:       {uniq_v:>8,}')
print(f'пользователей с хотя бы одним addtocart:  {uniq_a:>8,}  ({uniq_a/uniq_v*100:.1f}% от view)')
print(f'пользователей с хотя бы одной покупкой:   {uniq_t:>8,}  ({uniq_t/uniq_v*100:.1f}% от view)')

большинство пользователей только смотрят, до добавления в корзину доходит малая доля, до покупки — ещё меньше. конверсия низкая, что типично для e-commerce. для модели рекомендаций ориентируемся на addtocart + transaction как явный позитивный сигнал.

## Временная динамика

In [ ]:
events['date'] = events['timestamp'].dt.date
daily = events.groupby(['date', 'event']).size().unstack(fill_value=0)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
daily['view'].plot(ax=axes[0], color='steelblue')
axes[0].set_title('просмотры в день')
axes[0].set_ylabel('событий')

daily[['addtocart', 'transaction']].plot(ax=axes[1], color=['orange', 'green'])
axes[1].set_title('добавления в корзину и покупки в день')
axes[1].set_ylabel('событий')
axes[1].legend()
plt.tight_layout()
plt.show()

## Активность пользователей

In [ ]:
user_events = events.groupby('visitorid').size()
print('событий на пользователя:')
print(user_events.describe().round(1))
print()
print('% пользователей с 1 событием:', round((user_events == 1).mean() * 100, 1))
print('% пользователей с >=10 событиями:', round((user_events >= 10).mean() * 100, 1))

In [ ]:
fig, ax = plt.subplots()
user_events.clip(upper=50).value_counts().sort_index().plot(kind='bar', ax=ax, width=0.8)
ax.set_title('распределение числа событий на пользователя (обрезано до 50)')
ax.set_xlabel('событий')
ax.set_ylabel('пользователей')
plt.tight_layout()
plt.show()

типичный длинный хвост — большинство пользователей заходят один-два раза. это проблема холодного старта: у них нет истории для персонализации, придётся отдавать топ популярных товаров.

## Популярность товаров

In [ ]:
item_events = events.groupby('itemid').size().sort_values(ascending=False)
print('топ-10 товаров по числу событий:')
print(item_events.head(10))
print()
print('% товаров с >=10 событиями:', round((item_events >= 10).mean() * 100, 1))
print('% событий приходится на топ-1% товаров:',
      round(item_events.head(len(item_events) // 100).sum() / len(events) * 100, 1))

In [ ]:
fig, ax = plt.subplots()
ax.loglog(range(1, len(item_events) + 1), item_events.values, color='steelblue', lw=1)
ax.set_title('популярность товаров (log-log)')
ax.set_xlabel('ранг товара')
ax.set_ylabel('число событий')
plt.tight_layout()
plt.show()

степенной закон: небольшая часть товаров собирает большую часть взаимодействий. длинный хвост редких товаров будет сложнее рекомендовать.

## Категории

In [ ]:
cats = pd.read_csv(os.path.join(DATA_DIR, 'category_tree.csv'))
print('категорий всего:', len(cats))
print('корневых категорий (без родителя):', cats['parentid'].isna().sum())
cats.head()

In [ ]:
props_cols = ['itemid', 'property', 'value']
p1 = pd.read_csv(os.path.join(DATA_DIR, 'item_properties_part1.csv'), usecols=props_cols)
p2 = pd.read_csv(os.path.join(DATA_DIR, 'item_properties_part2.csv'), usecols=props_cols)
item_cats = pd.concat([p1[p1['property'] == 'categoryid'],
                       p2[p2['property'] == 'categoryid']])
item_cats = item_cats.groupby('itemid')['value'].last().reset_index()
item_cats.columns = ['itemid', 'categoryid']
item_cats['categoryid'] = pd.to_numeric(item_cats['categoryid'], errors='coerce')
print('товаров с категорией:', len(item_cats))
item_cats.head()

In [ ]:
# наиболее популярные категории по числу событий addtocart
add = events[events['event'] == 'addtocart'].merge(item_cats, on='itemid', how='left')
top_cats = add['categoryid'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 5))
top_cats.plot(kind='barh', ax=ax, color='orange')
ax.set_title('топ-15 категорий по числу добавлений в корзину')
ax.set_xlabel('addtocart событий')
plt.tight_layout()
plt.show()

## Выводы для модели

- данные с мая по сентябрь 2015 (~4 месяца)
- события: view (~97%), addtocart (~2%), transaction (~1%) — используем addtocart и transaction как позитивный сигнал
- ~85% пользователей с 1-2 событиями — холодный старт, отдаём топ популярных
- для остальных строим ALS на матрице user×item с весами: transaction=3, addtocart=2
- i2i (похожие товары) строим через косинусное сходство факторов ALS